# Config 2 time-resolved: VLS spectrum vs delay (multi-file)

Loads one or more config-2 time-resolved aggregates
(`compute_aggregates.py` with `MODE = "time_resolved"`) and plots
GMD-normalised VLS spectra against stage delay `z` — one column per
file (panels wrap to multiple rows past `MAX_COLS`).

Mirrors `cfg1_tr_etof_ion_vs_delay.ipynb`: specify the list in
`FILES` (and optionally matching `LABELS`), and every downstream
cell loops over a `runs` dict.

Figures produced:

1. **Unsummed**: 2D map of GMD-normalised VLS (`A / G`) vs delay,
   one panel per file. If `DELAY_BASELINE` is set, the delay-averaged
   spectrum from that window is subtracted from every delay slice
   (diverging colormap).
2. Same maps with a per-delay pixel-window baseline subtracted —
   the mean of the spectrum over `PIXEL_BASELINE` is subtracted
   from every pixel in the same delay slice. Removes a per-shot
   DC / dark offset. Stacks on top of the optional delay-baseline
   subtraction.
3. **Summed**: VLS integrated over `PIXEL_ROI` (1D overlay), one
   line per file vs delay.
4. **Per delay bin**: VLS spectra coloured by delay bin centre,
   one panel per file. Direct view of how the spectrum evolves
   along the scan.

`PIXEL_DOWNBIN` rebins the VLS pixel axis via a `(n_new, n_old)`
0/1 matrix that sums every `k` adjacent pixels. `k = 1` is a no-op.
The downbinning is applied to A and (in the load cell) the
matching `vls_pixels` axis so every figure sees a consistent grid.

In [ ]:
import sys
from pathlib import Path

_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize
from matplotlib.cm import ScalarMappable

import config
from compute_aggregates import load_aggregates
%matplotlib inline

## Parameters

`FILES` is the list of config-2 time-resolved aggregate paths to
compare. `LABELS` is an optional matching list; missing entries
fall back to each file's stem.

`GMD_BIN` is an integer applied to every file — every file must
have at least `GMD_BIN + 1` GMD bins.

`PIXEL_BASELINE` gives the pixel window (in source-pixel index, the
same units as `agg.vls_pixels`) used for plot 2's per-delay DC
baseline. `(None, None)` skips it.

`DELAY_BASELINE` (stage z units) selects a delay window whose
averaged spectrum is subtracted from every delay slice. `None`
skips. `PIXEL_ROI` is the pixel window summed in plot 3
(`(None, None)` integrates the full pixel range).

`PIXEL_DOWNBIN` is an integer factor that rebins the pixel axis by
summing every `k` adjacent pixels. Any tail pixels that don't fit a
full group are dropped.

In [ ]:
# --- input --------------------------------------------------------
FILES = [
    config.COMBINED_DIR / "test_config2_aggregates_tr.h5",
]
# Optional human-readable labels (one per file). Missing entries
# fall back to the file stem.
LABELS = []

# --- GMD bin selection -------------------------------------------
GMD_BIN  = 1            # integer index, applied to every file

# --- Pixel downbinning (matrix transformation) -------------------
# Sum every k adjacent pixels. 1 = no downbinning.
PIXEL_DOWNBIN = 1

# --- Pixel-window baseline ("two pixels") ------------------------
# Mean over this pixel window is subtracted from every pixel in the
# same delay slice. Units = source-pixel index.
PIXEL_BASELINE = (None, None)          # e.g. (0, 50)

# --- Delay-window baseline ---------------------------------------
# Mean spectrum over this delay window is subtracted from every
# delay slice. Set to None to skip. Units = stage z.
DELAY_BASELINE = None                  # e.g. (-15000.0, -10000.0)

# --- Pixel ROI for plot 3 ----------------------------------------
# Pixel window summed for the 1D delay trace.
PIXEL_ROI = (None, None)               # e.g. (380, 500)

# --- Plot 4 line selection ---------------------------------------
# Subsample delay bins to keep the per-delay line plot readable.
# None plots every delay bin. Otherwise sample evenly to this count.
N_DELAY_LINES = 20

# --- cosmetics ---------------------------------------------------
PCT_LOW, PCT_HIGH = 1.0, 99.0          # robust colour-scale limits
MAX_COLS          = 3                  # subplot wrap threshold

print(f"files to compare: {len(FILES)}")
for p in FILES:
    print(f"  {p}")

## Load and normalise

Each file is loaded and validated as a config-2 time-resolved
aggregate. For the chosen `GMD_BIN`:

1. The pixel axis is downbinned via a `(n_new, n_old)` matrix `M`
   of ones that sums every `k` adjacent pixels. Applied to the
   first moments (A) as `A @ Mᵀ`.
2. The downbinned per-shot spectrum is GMD-normalised
   (`A_db / G[:, None]`) so each map is in counts-per-shot per uJ.

`runs` is a dict keyed by label; every downstream cell iterates
over `runs.items()`.

In [ ]:
def _downbin_matrix(n_old, k):
    """(n_new, n_old) 0/1 matrix that sums every k consecutive bins."""
    if k <= 0:
        raise ValueError(f"downbin factor must be >= 1, got {k}")
    n_new = n_old // k
    if n_new == 0:
        raise ValueError(f"downbin factor {k} > n_old={n_old}")
    M = np.zeros((n_new, n_old), dtype=np.float64)
    for i in range(n_new):
        M[i, i * k:(i + 1) * k] = 1.0
    return M


def _label_for(i, path):
    if i < len(LABELS) and LABELS[i]:
        return LABELS[i]
    return Path(path).stem


def load_and_prepare(path):
    agg = load_aggregates(path)
    if agg.config != 2:
        raise ValueError(
            f"{Path(path).name}: expected config=2, got config={agg.config}"
        )
    if agg.mode != "time_resolved":
        raise ValueError(
            f"{Path(path).name}: expected mode='time_resolved', "
            f"got mode={agg.mode!r}"
        )
    if not (0 <= GMD_BIN < agg.n_gmd_bins):
        raise ValueError(
            f"{Path(path).name}: GMD_BIN={GMD_BIN} out of range "
            f"[0, {agg.n_gmd_bins})"
        )

    M = _downbin_matrix(agg.n_pixels, PIXEL_DOWNBIN)

    # A: shape (n_z, n_pixels). Downbin via A @ Mᵀ along pixel axis.
    A_db = agg.A[GMD_BIN] @ M.T

    with np.errstate(invalid="ignore", divide="ignore"):
        A = A_db / agg.G[GMD_BIN][:, None]

    # Downbinned pixel axis: take every k-th pixel-axis entry as the
    # bin centre representative (the matrix groups k adjacent pixels).
    pixel_db = agg.vls_pixels[: (agg.n_pixels // PIXEL_DOWNBIN)
                              * PIXEL_DOWNBIN].reshape(-1, PIXEL_DOWNBIN).mean(axis=1)

    gmd_label = (f"GMD [{agg.gmd_edges[GMD_BIN]:.3g}, "
                 f"{agg.gmd_edges[GMD_BIN + 1]:.3g}) uJ")
    return {
        "agg": agg,
        "A": A,
        "pixel_axis":  pixel_db,
        "z_edges":     agg.z_edges,
        "gmd_label":   gmd_label,
        "n_pixels":    A.shape[-1],
        "n_z":         agg.n_z_bins,
        "n_per_bin":   agg.n_per_bin[GMD_BIN],
        "M":           M,
    }


runs = {_label_for(i, p): load_and_prepare(p) for i, p in enumerate(FILES)}

for label, run in runs.items():
    print(f"{label}:")
    print(f"  GMD        : {run['gmd_label']}")
    print(f"  n_z        : {run['n_z']}")
    print(f"  n_pixels   : {run['n_pixels']}  (downbin x{PIXEL_DOWNBIN})")
    print(f"  shots/bin  : min={run['n_per_bin'].min()}, "
          f"max={run['n_per_bin'].max()}, "
          f"total={run['n_per_bin'].sum()}")

## Subtraction + plotting helpers

`_subtract_pixel_baseline`: per-delay mean over a pixel window,
subtracted from every pixel in the same delay slice.

`_subtract_delay_baseline`: mean spectrum over a delay window,
subtracted from every delay slice.

`_plot_panel` paints one map (pcolormesh) — either viridis (raw)
or symmetric diverging RdBu_r (after any subtraction).

`_file_grid` allocates the wrapped multi-file axes grid.

`_integrate_roi_vs_delay` applies the subtractions and returns the
per-delay integral over a pixel ROI — backs plot 3.

In [ ]:
def _window_indices_axis(axis, lo, hi):
    """Index range [i_lo, i_hi) of axis points whose values fall in [lo, hi).
    Used for the pixel axis (which is a 1D array of centres, not edges).
    """
    axis = np.asarray(axis)
    if lo is None:
        lo = axis[0]
    if hi is None:
        hi = axis[-1] + np.finfo(float).eps
    sel = (axis >= lo) & (axis < hi)
    idx = np.where(sel)[0]
    if idx.size == 0:
        raise ValueError(
            f"empty window [{lo}, {hi}) for axis spanning "
            f"[{axis[0]}, {axis[-1]}]"
        )
    return int(idx[0]), int(idx[-1] + 1)


def _window_indices_edges(edges, lo, hi):
    """Index range [i_lo, i_hi) of bins whose centres fall in [lo, hi)."""
    cent = 0.5 * (edges[:-1] + edges[1:])
    if lo is None:
        lo = cent[0]
    if hi is None:
        hi = cent[-1] + np.finfo(float).eps
    sel = (cent >= lo) & (cent < hi)
    idx = np.where(sel)[0]
    if idx.size == 0:
        raise ValueError(
            f"empty window [{lo}, {hi}) for edges spanning "
            f"[{edges[0]}, {edges[-1]})"
        )
    return int(idx[0]), int(idx[-1] + 1)


def _subtract_pixel_baseline(spec, pixel_axis, pixel_window):
    if pixel_window is None or pixel_window == (None, None):
        return spec
    lo, hi = pixel_window
    if lo is None and hi is None:
        return spec
    i0, i1 = _window_indices_axis(pixel_axis, lo, hi)
    base = np.nanmean(spec[:, i0:i1], axis=1, keepdims=True)
    return spec - base


def _subtract_delay_baseline(spec, z_edges, delay_window):
    if delay_window is None:
        return spec
    lo, hi = delay_window
    if lo is None and hi is None:
        return spec
    j0, j1 = _window_indices_edges(z_edges, lo, hi)
    base = np.nanmean(spec[j0:j1, :], axis=0, keepdims=True)
    return spec - base


def _robust_limits(arr, pct_lo, pct_hi, symmetric=False):
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return -1.0, 1.0
    if symmetric:
        lim = float(np.nanpercentile(np.abs(finite), pct_hi))
        if lim <= 0:
            lim = float(np.nanmax(np.abs(finite))) or 1.0
        return -lim, lim
    return (float(np.nanpercentile(finite, pct_lo)),
            float(np.nanpercentile(finite, pct_hi)))


def _pixel_edges(pixel_axis):
    """Half-step edges around a (possibly non-uniform) pixel axis,
    for pcolormesh."""
    p = np.asarray(pixel_axis, dtype=np.float64)
    mid = 0.5 * (p[:-1] + p[1:])
    return np.concatenate([[2 * p[0] - mid[0]], mid, [2 * p[-1] - mid[-1]]])


def _plot_panel(ax, spec, x_edges, y_edges, title, xlabel, cbar_label,
                symmetric):
    if symmetric:
        vmin, vmax = _robust_limits(spec, PCT_LOW, PCT_HIGH, symmetric=True)
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        pcm = ax.pcolormesh(x_edges, y_edges, spec, cmap="RdBu_r",
                            norm=norm, shading="auto")
    else:
        vmin, vmax = _robust_limits(spec, PCT_LOW, PCT_HIGH)
        pcm = ax.pcolormesh(x_edges, y_edges, spec, cmap="viridis",
                            vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    cbar = ax.figure.colorbar(pcm, ax=ax)
    cbar.set_label(cbar_label)
    return pcm


def _tag(window, name):
    if window is None or window == (None, None):
        return f"no {name}"
    return f"{name} [{window[0]}, {window[1]})"


def _delay_tag():
    return (f", delay [{DELAY_BASELINE[0]}, {DELAY_BASELINE[1]})"
            if DELAY_BASELINE is not None else "")


def _file_grid(n_metric_rows, n_files, panel_size=(6.0, 4.0),
               max_cols=None, sharey="row"):
    if max_cols is None:
        max_cols = MAX_COLS
    cols = min(max_cols, n_files)
    file_rows = (n_files + cols - 1) // cols
    fig, axes = plt.subplots(
        n_metric_rows * file_rows, cols,
        figsize=(panel_size[0] * cols,
                 panel_size[1] * n_metric_rows * file_rows),
        sharey=sharey, constrained_layout=True, squeeze=False,
    )

    def ax_at(k, m):
        fr, fc = divmod(k, cols)
        return axes[fr * n_metric_rows + m, fc]

    def hide_unused():
        for k in range(n_files, file_rows * cols):
            fr, fc = divmod(k, cols)
            for m in range(n_metric_rows):
                axes[fr * n_metric_rows + m, fc].axis("off")

    return fig, axes, ax_at, hide_unused


def _integrate_roi_vs_delay(spec, pixel_axis, z_edges, pixel_baseline,
                            pixel_roi, n_pixels):
    """Subtract baselines, integrate `spec` over `pixel_roi`, return
    (z_centres, integrated, (i0, i1)).
    """
    s = _subtract_pixel_baseline(spec, pixel_axis, pixel_baseline)
    s = _subtract_delay_baseline(s, z_edges, DELAY_BASELINE)

    lo, hi = pixel_roi
    if lo is None and hi is None:
        i0, i1 = 0, n_pixels
    else:
        i0, i1 = _window_indices_axis(pixel_axis, lo, hi)

    z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
    integrated = s[:, i0:i1].sum(axis=1)
    return z_cent, integrated, (i0, i1)

## Plot 1 — unsummed: GMD-normalised VLS vs delay (2D map)

One panel per file. Rows = delay z; columns = pixel index. Colour =
`A / G` (counts per shot per uJ). If `DELAY_BASELINE` is set, the
delay-averaged spectrum is subtracted (diverging colormap).

In [ ]:
sym1 = DELAY_BASELINE is not None
suffix = "  - delay-baseline" if sym1 else ""
n_files = len(runs)

fig, axes, ax_at, hide_unused = _file_grid(1, n_files,
                                           panel_size=(6, 5))
for k, (label, run) in enumerate(runs.items()):
    A_p1 = _subtract_delay_baseline(run["A"], run["z_edges"], DELAY_BASELINE)
    pix_edges = _pixel_edges(run["pixel_axis"])
    _plot_panel(ax_at(k, 0), A_p1, pix_edges, run["z_edges"],
                f"{label}  -  VLS{suffix}",
                "VLS pixel", "A / G (counts/shot/uJ)", sym1)

hide_unused()
for ax in axes[:, 0]:
    ax.set_ylabel("stage z (arb.)")
gmd_labels = " | ".join(sorted({r["gmd_label"] for r in runs.values()}))
fig.suptitle(f"GMD-normalised VLS vs delay  -  {gmd_labels}")
plt.show()

## Plot 2 — unsummed with pixel-window baseline subtracted

For each delay slice the mean of the spectrum over `PIXEL_BASELINE`
is subtracted from every pixel. Use this to remove a delay-dependent
DC offset (dark / scatter floor). If `DELAY_BASELINE` is also set,
the delay-averaged spectrum is subtracted on top.

In [ ]:
n_files = len(runs)

fig, axes, ax_at, hide_unused = _file_grid(1, n_files,
                                           panel_size=(6, 5))
for k, (label, run) in enumerate(runs.items()):
    A_p2 = _subtract_pixel_baseline(run["A"], run["pixel_axis"],
                                    PIXEL_BASELINE)
    A_p2 = _subtract_delay_baseline(A_p2, run["z_edges"], DELAY_BASELINE)
    pix_edges = _pixel_edges(run["pixel_axis"])
    _plot_panel(
        ax_at(k, 0), A_p2, pix_edges, run["z_edges"],
        f"{label}  -  VLS\n{_tag(PIXEL_BASELINE, 'pixel baseline')}{_delay_tag()}",
        "VLS pixel", "(A - baseline) / G", symmetric=True,
    )

hide_unused()
for ax in axes[:, 0]:
    ax.set_ylabel("stage z (arb.)")
gmd_labels = " | ".join(sorted({r["gmd_label"] for r in runs.values()}))
fig.suptitle(f"VLS, pixel-baseline subtracted  -  {gmd_labels}")
plt.show()

## Plot 3 — summed: VLS integrated over `PIXEL_ROI` vs delay (1D overlay)

For each file, sum the (possibly baseline-subtracted) GMD-normalised
VLS spectrum over `PIXEL_ROI` along the pixel axis, giving a 1D
curve of integrated VLS vs delay. All files are overlaid on a
single panel.

In [ ]:
sym_roi = (PIXEL_BASELINE != (None, None)) or (DELAY_BASELINE is not None)

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)

for label, run in runs.items():
    z_cent, integrated, (i0, i1) = _integrate_roi_vs_delay(
        run["A"], run["pixel_axis"], run["z_edges"],
        PIXEL_BASELINE, PIXEL_ROI, run["n_pixels"],
    )
    print(f"{label}: pixel ROI bins [{i0}, {i1}) -> "
          f"[{run['pixel_axis'][i0]:.1f}, {run['pixel_axis'][i1 - 1]:.1f}] pixels")
    ax.plot(z_cent, integrated, lw=1.4, label=label)

ax.set_xlabel("stage z (arb.)")
ax.set_ylabel(
    "Σ (A - baseline) / G over ROI" if sym_roi
    else "Σ A / G over ROI  (counts/shot/uJ)"
)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("VLS integrated over pixel ROI vs delay")
plt.show()

## Plot 4 — per-delay-bin VLS spectra (one panel per file)

For each file, plot one VLS spectrum per delay bin coloured by the
bin's z centre (viridis). `N_DELAY_LINES` evenly subsamples the
delay axis to keep the plot readable; set to `None` to draw every
bin. Subtractions configured above are applied first.

In [ ]:
n_files = len(runs)
sym4 = (PIXEL_BASELINE != (None, None)) or (DELAY_BASELINE is not None)

fig, axes, ax_at, hide_unused = _file_grid(
    1, n_files, panel_size=(6, 5), sharey=True,
)

for k, (label, run) in enumerate(runs.items()):
    A4 = _subtract_pixel_baseline(run["A"], run["pixel_axis"], PIXEL_BASELINE)
    A4 = _subtract_delay_baseline(A4, run["z_edges"], DELAY_BASELINE)

    z_cent = 0.5 * (run["z_edges"][:-1] + run["z_edges"][1:])
    n_z = z_cent.size
    if N_DELAY_LINES is None or N_DELAY_LINES >= n_z:
        idx = np.arange(n_z)
    else:
        idx = np.linspace(0, n_z - 1, N_DELAY_LINES).round().astype(int)

    norm = Normalize(vmin=float(z_cent.min()), vmax=float(z_cent.max()))
    ax = ax_at(k, 0)
    for j in idx:
        spec_j = A4[j]
        if not np.isfinite(spec_j).any():
            continue
        ax.plot(run["pixel_axis"], spec_j,
                color=plt.cm.viridis(norm(z_cent[j])), lw=1.0)
    sm = ScalarMappable(norm=norm, cmap="viridis"); sm.set_array([])
    fig.colorbar(sm, ax=ax, label="delay z (arb.)")
    ax.set_xlabel("VLS pixel")
    ax.set_ylabel(
        "(A - baseline) / G" if sym4 else "A / G (counts/shot/uJ)"
    )
    ax.set_title(f"{label}  ({len(idx)} of {n_z} delay bins)")
    ax.grid(alpha=0.3)

hide_unused()
gmd_labels = " | ".join(sorted({r["gmd_label"] for r in runs.values()}))
fig.suptitle(f"VLS per delay bin  -  {gmd_labels}")
plt.show()